# Evaluasi Model Klasifikasi BISINDO (SVM RBF)

Notebook ini mengevaluasi model SVM yang benar-benar dipakai server web, bukan data contoh.
Model dimuat dari `ml/models/svm_rbf_bisindo.joblib`, lalu diuji pada data uji yang sama
dengan yang dipakai `train_svm.py` (test_size 0.2, random_state 42).

Isi notebook:
1. Menghitung accuracy, precision, recall, dan F1-Score
2. Membuat confusion matrix 26x26
3. Menganalisis huruf yang paling sering salah terdeteksi
4. Memverifikasi bahwa hasil notebook sama dengan `metrics.json`

> Catatan penting: dataset punya keterbatasan yang memengaruhi cara membaca akurasi.
> Lihat bagian terakhir sebelum mengutip angka dari notebook ini.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# Notebook ada di ml/notebooks, sedangkan modul ada di ml/src.
ML_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = ML_DIR / "src"
MODELS_DIR = ML_DIR / "models"
OUTPUT_DIR = MODELS_DIR / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from dataset.data_loader import FEATURE_COUNT, load_real_data  # noqa: E402
from training.train_svm import LABEL_MAP_FILENAME, MODEL_FILENAME  # noqa: E402

sns.set_theme(style="whitegrid")
print("Modul siap. Folder model:", MODELS_DIR)

Modul siap. Folder model: C:\Semester_5\PBL\bisindo\ml\models


In [2]:
model = joblib.load(MODELS_DIR / MODEL_FILENAME)
label_map = json.loads((MODELS_DIR / LABEL_MAP_FILENAME).read_text(encoding="utf-8"))
class_names = [str(name) for name in label_map["classes"]]

# Split yang sama dengan train_svm.py: test_size 0.2, random_state 42.
_, x_test, _, y_test, _ = load_real_data()
y_pred = model.predict(x_test)

y_true_letters = np.array(class_names)[y_test]
y_pred_letters = np.array(class_names)[y_pred]

print(f"Model    : {type(model).__name__} {list(dict(model.steps))}")
print(f"Fitur    : {model.named_steps['svm'].n_features_in_} (kontrak: {FEATURE_COUNT})")
print(f"Kelas    : {len(class_names)}")
print(f"Data uji : {len(y_test)} baris")

Model    : Pipeline ['scaler', 'svm']
Fitur    : 127 (kontrak: 127)
Kelas    : 26
Data uji : 1579 baris


In [3]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

benar = int((y_test == y_pred).sum())
salah = int((y_test != y_pred).sum())

print(f"Akurasi     : {accuracy * 100:.2f}%")
print(f"Precision   : {precision:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"F1 weighted : {f1:.4f}")
print(f"Benar       : {benar} dari {len(y_test)}")
print(f"Salah       : {salah}")
print()
print("--- Laporan Klasifikasi per Huruf ---")
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

Akurasi     : 97.91%
Precision   : 0.9797
Recall      : 0.9791
F1 weighted : 0.9791
Benar       : 1546 dari 1579
Salah       : 33

--- Laporan Klasifikasi per Huruf ---
              precision    recall  f1-score   support

           A       1.00      0.96      0.98        45
           B       0.98      0.90      0.94        61
           C       0.93      0.98      0.96        58
           D       0.96      1.00      0.98        65
           E       0.97      0.98      0.98        62
           F       1.00      0.98      0.99        57
           G       0.94      1.00      0.97        58
           H       0.98      0.96      0.97        56
           I       1.00      1.00      1.00        71
           J       1.00      0.99      0.99        68
           K       1.00      0.98      0.99        52
           L       0.97      0.97      0.97        65
           M       0.97      1.00      0.99        71
           N       0.99      0.97      0.98        70
           O       1

In [4]:
matrix = confusion_matrix(y_test, y_pred)

figure, axis = plt.subplots(figsize=(15, 13))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={"label": "Jumlah sampel"},
    ax=axis,
)
axis.set_title(
    f"Confusion Matrix SVM RBF\nAkurasi {accuracy * 100:.2f}% | F1 {f1:.4f}",
    fontsize=15,
    fontweight="bold",
)
axis.set_ylabel("Label Sebenarnya")
axis.set_xlabel("Label Prediksi")
figure.tight_layout()

figure.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Disimpan ke:", OUTPUT_DIR / "confusion_matrix.png")

Disimpan ke: C:\Semester_5\PBL\bisindo\ml\models\evaluation\confusion_matrix.png


In [5]:
pasangan = []
for i, huruf_asli in enumerate(class_names):
    for j, huruf_tebak in enumerate(class_names):
        if i != j and matrix[i, j] > 0:
            pasangan.append((int(matrix[i, j]), huruf_asli, huruf_tebak))

pasangan.sort(key=lambda item: (-item[0], item[1]))

print(f"Total kesalahan: {salah} dari {len(y_test)} sampel uji")
print()
print("Top 5 huruf paling sering tertukar:")
for urutan, (jumlah, huruf_asli, huruf_tebak) in enumerate(pasangan[:5], start=1):
    print(
        f"{urutan}. Huruf '{huruf_asli}' terdeteksi sebagai "
        f"'{huruf_tebak}' sebanyak {jumlah} kali"
    )

print()
print("F1 terendah per huruf:")
laporan = classification_report(
    y_test, y_pred, target_names=class_names, output_dict=True, zero_division=0
)
for huruf in sorted(class_names, key=lambda name: laporan[name]["f1-score"])[:5]:
    print(f"  {huruf}: F1 {laporan[huruf]['f1-score']:.4f}")

Total kesalahan: 33 dari 1579 sampel uji

Top 5 huruf paling sering tertukar:
1. Huruf 'P' terdeteksi sebagai 'C' sebanyak 3 kali
2. Huruf 'B' terdeteksi sebagai 'E' sebanyak 2 kali
3. Huruf 'N' terdeteksi sebagai 'M' sebanyak 2 kali
4. Huruf 'S' terdeteksi sebagai 'Q' sebanyak 2 kali
5. Huruf 'A' terdeteksi sebagai 'L' sebanyak 1 kali

F1 terendah per huruf:
  Y: F1 0.9213
  B: F1 0.9402
  C: F1 0.9580
  S: F1 0.9630
  P: F1 0.9635


In [6]:
tersimpan = json.loads((MODELS_DIR / "metrics.json").read_text(encoding="utf-8"))

print("Perbandingan notebook vs metrics.json")
print(f"  akurasi notebook : {accuracy:.6f}")
print(f"  akurasi tersimpan: {tersimpan['accuracy']:.6f}")
print(f"  selisih          : {abs(accuracy - tersimpan['accuracy']):.10f}")
print()

if abs(accuracy - tersimpan["accuracy"]) < 1e-9:
    print("Cocok. Notebook dan artefak model sinkron.")
else:
    print("TIDAK COCOK. Model kemungkinan sudah dilatih ulang.")
    print("Jalankan ulang: python ml/src/training/train_svm.py --no-cv")

Perbandingan notebook vs metrics.json
  akurasi notebook : 0.979101
  akurasi tersimpan: 0.979101
  selisih          : 0.0000000000

Cocok. Notebook dan artefak model sinkron.


## Cara membaca hasil ini

**Angka 97,91% bukan perkiraan performa di dunia nyata.**

Dataset berisi 7.893 baris, tetapi hanya 668 nama berkas unik. Nama berkas hasil
augmentasi dipakai ulang di banyak kelas, misalnya `augmented_image_10.jpg` muncul
pada 12 kelas berbeda. Akibatnya pembagian data latih dan uji tidak bisa dilakukan
per gambar asal, hanya per baris.

Pada split 20% yang dipakai di sini, ada **436 nama berkas yang muncul di data latih
sekaligus data uji**. Model bisa mengenali versi augmentasi dari gambar yang sama,
sehingga akurasi cenderung lebih tinggi daripada performa pada gambar baru.

Angka ini sah untuk protokol "split acak per baris", dan itulah yang harus ditulis
di laporan. Jangan menuliskannya sebagai akurasi pengenalan gambar baru.

Detail lengkap tersimpan di `ml/models/metrics.json` pada kunci `dataset_warning`
dan `filename_overlap`.